In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error

f = r"D:\PROJECTS\YAANENDRIYA\data\ALiMU_working\real_sensor_data_dyn_01_MPU9250_VN100.txt"

df = pd.read_csv(f)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print(df.head())

Shape: (14993, 18)
Columns: ['MPU9250_ax', 'MPU9250_ay', 'MPU9250_az', 'MPU9250_gx', 'MPU9250_gy', 'MPU9250_gz', 'MPU9250_mx', 'MPU9250_my', 'MPU9250_mz', 'VN100_ax', 'VN100_ay', 'VN100_az', 'VN100_gx', 'VN100_gy', 'VN100_gz', 'VN100_mx', 'VN100_my', 'VN100_mz']
   MPU9250_ax  MPU9250_ay  MPU9250_az  MPU9250_gx  MPU9250_gy  MPU9250_gz  \
0       0.746      -0.108     -10.578       0.000       0.016       0.013   
1       0.842      -0.105     -10.585      -0.019       0.011       0.020   
2       0.831      -0.186     -10.501      -0.011       0.011       0.018   
3       0.822      -0.206     -10.439      -0.011       0.005       0.005   
4       0.818      -0.150     -10.461       0.012      -0.008      -0.007   

   MPU9250_mx  MPU9250_my  MPU9250_mz  VN100_ax  VN100_ay  VN100_az  VN100_gx  \
0      -0.213      -0.016       0.301     0.635    -0.124    -9.299 -0.010245   
1      -0.217      -0.026       0.297     0.635    -0.124    -9.299 -0.010245   
2      -0.217      -0.026      

In [3]:
print(df.dtypes)
print(df.describe())

MPU9250_ax    float64
MPU9250_ay    float64
MPU9250_az    float64
MPU9250_gx    float64
MPU9250_gy    float64
MPU9250_gz    float64
MPU9250_mx    float64
MPU9250_my    float64
MPU9250_mz    float64
VN100_ax      float64
VN100_ay      float64
VN100_az      float64
VN100_gx      float64
VN100_gy      float64
VN100_gz      float64
VN100_mx      float64
VN100_my      float64
VN100_mz      float64
dtype: object
         MPU9250_ax    MPU9250_ay    MPU9250_az    MPU9250_gx    MPU9250_gy  \
count  14993.000000  14993.000000  14993.000000  14993.000000  14993.000000   
mean      -2.771713     -0.753265     -3.489652     -0.044957     -0.023632   
std        5.750356      5.118273      6.592692      1.191913      1.378058   
min      -18.556000    -19.999001    -19.999001     -3.179000     -3.217000   
25%       -7.359000     -3.418000     -9.851000     -0.479000     -0.537000   
50%       -1.249000     -0.132000     -3.069000     -0.005000      0.003000   
75%        0.938000      1.307000    

In [4]:
# Real sensor error relative to VN100 reference

for axis in ["x", "y", "z"]:
    df[f"error_{axis}"] = (
        df[f"MPU9250_a{axis}"] -
        df[f"VN100_a{axis}"]
    )

feature_cols = [
    "MPU9250_ax",
    "MPU9250_ay",
    "MPU9250_az"
]

target_cols = [
    "error_x",
    "error_y",
    "error_z"
]

print(df[target_cols].describe())

            error_x       error_y       error_z
count  14993.000000  14993.000000  14993.000000
mean      -0.169836     -0.082836     -0.850430
std        0.557688      0.590671      0.728935
min      -17.394000     -5.738000    -15.839001
25%       -0.369000     -0.286000     -1.167000
50%       -0.091001     -0.040000     -0.862000
75%        0.155000      0.186000     -0.535000
max        3.338000     20.116000     10.354000


In [5]:
X = df[feature_cols].values
Y = df[target_cols].values

split = int(0.8 * len(df))

X_train = X[:split]
X_test = X[split:]

Y_train = Y[:split]
Y_test = Y[split:]

scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = Ridge(alpha=1.0)

model.fit(X_train_s, Y_train)

Y_pred = model.predict(X_test_s)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (11994, 3)
Testing : (2999, 3)


In [6]:
for i, axis in enumerate(["X", "Y", "Z"]):

    baseline = np.full(
        len(Y_test),
        Y_train[:, i].mean()
    )

    baseline_rmse = np.sqrt(
        mean_squared_error(
            Y_test[:, i],
            baseline
        )
    )

    ml_rmse = np.sqrt(
        mean_squared_error(
            Y_test[:, i],
            Y_pred[:, i]
        )
    )

    ml_mae = mean_absolute_error(
        Y_test[:, i],
        Y_pred[:, i]
    )

    improvement = (
        1 - ml_rmse / baseline_rmse
    ) * 100

    print(f"\n{axis}-axis")
    print(f"Baseline RMSE : {baseline_rmse:.6f}")
    print(f"ML RMSE       : {ml_rmse:.6f}")
    print(f"ML MAE        : {ml_mae:.6f}")
    print(f"Improvement   : {improvement:.2f}%")


X-axis
Baseline RMSE : 0.532112
ML RMSE       : 0.496385
ML MAE        : 0.256921
Improvement   : 6.71%

Y-axis
Baseline RMSE : 0.647094
ML RMSE       : 0.593405
ML MAE        : 0.294246
Improvement   : 8.30%

Z-axis
Baseline RMSE : 0.720256
ML RMSE       : 0.592094
ML MAE        : 0.339251
Improvement   : 17.79%


In [7]:
print("\nReal residual mean:")
print(df[target_cols].mean())

print("\nReal residual STD:")
print(df[target_cols].std())

for axis in ["x", "y", "z"]:
    rolling = df[f"error_{axis}"].rolling(1000).mean()

    print(
        f"{axis.upper()} rolling mean range:",
        round(rolling.min(), 6),
        "to",
        round(rolling.max(), 6)
    )


Real residual mean:
error_x   -0.169836
error_y   -0.082836
error_z   -0.850430
dtype: float64

Real residual STD:
error_x    0.557688
error_y    0.590671
error_z    0.728935
dtype: float64
X rolling mean range: -0.545779 to 0.17817
Y rolling mean range: -0.31725 to 0.169141
Z rolling mean range: -1.071509 to -0.59757
